# Archived research notebook

Original filename: `preprocessing forth伪影检测和坏道.ipynb`  
Purpose: Detect artifacts and bad EEG channels.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
#运行须知！！这个会输出一整个文件夹的PED图像 所以你看几个知道没问题后就可以点暂停停止输出了不然可能会程序崩溃或者电脑死机
import os
import mne
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
#matplotlib.use('TkAgg')引用这一行代码可以解决matplotlib无法和jupyter交互的问题
import matplotlib.pyplot as plt
import warnings

# 忽略文件名不符合 MNE 命名规范的警告
warnings.filterwarnings("ignore", category=RuntimeWarning, message="This filename.*does not conform to MNE naming conventions.")

# 定义要处理的文件夹路径
folder_path = r"data/chb-mit/chb01"

# 定义基础频率，对于美国的 CHB - MIT 数据集，设置为 60Hz
base_freq = 60
# 定义谐波次数
harmonic_count = 4

# 遍历文件夹中的所有文件
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith('_raw_no_proj.fif'):
            # 构建文件的完整路径
            fif_path = os.path.join(root, file)
            try:
                # 读取 fif 文件
                raw = mne.io.read_raw_fif(fif_path, preload=True)

                # 获取数据的采样频率
                sfreq = raw.info['sfreq']
                # 计算最大可分析频率，确保不超过采样频率的一半
                fmax = min(250, sfreq / 2)
                print(f"文件 {file} 的采样频率为 {sfreq}Hz，fmax 为 {fmax}Hz")

                # 计算并绘制功率谱密度
                fig = raw.compute_psd(tmax=np.inf, fmax=fmax).plot(
                    average=True, amplitude=False, picks="data", exclude="bads"
                )

                print(f"fig.axes 的长度为 {len(fig.axes)}")
                for i, ax in enumerate(fig.axes):
                    print(f"子图 {i} 的线条数量为 {len(ax.lines)}")

                # 生成要检测的频率列表
                frequencies = [base_freq * i for i in range(1, harmonic_count + 1)]

                # 在指定频率处添加箭头
                for ax in fig.axes:
                    freqs = ax.lines[-1].get_xdata()
                    psds = ax.lines[-1].get_ydata()

                    print(f"freqs 的长度为 {len(freqs)}")
                    print(f"psds 的长度为 {len(psds)}")

                    for freq in frequencies:
                        if freq <= fmax:
                            idx = np.argmin(np.abs(np.array(freqs) - freq))
                            ax.arrow(
                                x=freqs[idx],
                                y=psds[idx] + 18,
                                dx=0,
                                dy=-12,
                                color="red",
                                width=0.1,
                                head_width=3,
                                length_includes_head=True,
                            )

                # 显示图形
                plt.show()

            except Exception as e:
                print(f"处理 {file} 时出错: {e}")

print("所有文件处理完成")


In [ ]:
print(raw.info["bads"])
#从运行结果上看显然他没有标注坏道且文件里也没有对此进行说明所以我没有就坏道的问题进行后续处理